# 00 · Kiểm tra môi trường Kaggle và chốt requirements.txt (ticket #2)

Chắc chắn implicit (ALS), gensim (item2vec) và PyTorch (SASRec, mix học) chạy được trên image Kaggle hiện tại trước khi viết mô hình (#14–#16, #23). Notebook cũng chạy bộ test của repo để kiểm tra `src/` trên Python và các gói của Kaggle.

**Input:** không cần dataset; đọc `requirements.txt` từ repo vừa clone.

**Output:** không ghi file. Ô cuối in khối `requirements.txt` theo các gói đang cài, đánh dấu gói lệch phiên bản đã chốt.

Bật **Internet** (để clone repo và `pip install`). Chỉ cần CPU. Chạy **Run All**: ô nào lỗi thì Run All dừng ở đó.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #2 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
REPO = "/kaggle/working/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
# Phiên bản đã chốt (requirements.txt, áp marker theo Python đang chạy) so với phiên bản đang cài
import importlib, importlib.metadata as metadata, platform
from pathlib import Path
from packaging.requirements import Requirement

PYTHON = sys.version.split()[0]
pins = {}
for line in Path(REPO, "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if not line:
        continue
    req = Requirement(line)
    if req.marker is None or req.marker.evaluate():
        pins[req.name] = str(req.specifier).removeprefix("==")

def installed(name: str) -> str | None:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

print("Python", PYTHON, "|", platform.platform())
for name, pin in pins.items():
    print(f"{name:10s} chốt {pin:8s} đang có {installed(name) or 'CHƯA CÓ'}")

In [ ]:
# Cài đúng phiên bản đã chốt cho gói còn thiếu (thường là implicit). Lỗi thì dừng ở đây: chép log về để ghi vào README.
import subprocess

missing = [f"{name}=={pin}" for name, pin in pins.items() if installed(name) is None]
print("cài thêm:", missing or "không có")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
importlib.invalidate_caches()

In [ ]:
# Bộ test của repo: src/ phải chạy được trên Python + numpy/pandas của Kaggle. Test hỏng thì ô này báo lỗi.
tests = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, capture_output=True, text=True)
print("\n".join(tests.stdout.strip().splitlines()[-3:]))
assert tests.returncode == 0, "LỖI: bộ test của repo không qua trên Kaggle"

In [ ]:
# ALS (implicit) trên ma trận playlist × bài đồ chơi, ép chạy CPU
import numpy as np
import scipy.sparse as sp
from implicit.cpu.als import AlternatingLeastSquares

playlist_items = sp.random(300, 120, density=0.05, format="csr", random_state=0, dtype=np.float32)
playlist_items.data[:] = 1.0
als = AlternatingLeastSquares(factors=16, iterations=5, regularization=0.01, random_state=0)
als.fit(playlist_items, show_progress=False)
ids, scores = als.recommend(0, playlist_items[0], N=5)
assert als.item_factors.shape == (120, 16)
assert len(ids) == 5 and not set(ids) & set(playlist_items[0].indices), "LỖI: ALS gợi ý lại bài đã có"
print("ALS OK | item_factors", als.item_factors.shape, "| top-5 của playlist 0:", ids.tolist())

In [ ]:
# item2vec: Word2Vec (gensim) trên chuỗi bài của từng playlist đồ chơi
from gensim.models import Word2Vec

rng = np.random.default_rng(0)
playlists = [[f"t{int(i)}" for i in rng.integers(0, 120, 20)] for _ in range(300)]
w2v = Word2Vec(playlists, vector_size=16, window=5, min_count=1, sg=1, epochs=5, workers=1, seed=0)
nearest = [t for t, _ in w2v.wv.most_similar("t0", topn=3)]
assert w2v.wv.vector_size == 16 and len(nearest) == 3 and "t0" not in nearest
print("Word2Vec OK | số bài", len(w2v.wv), "| gần t0 nhất:", nearest)

In [ ]:
# PyTorch: vài bước train một mô hình nhỏ (embedding + linear, Adam), loss phải giảm
import torch

torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Embedding(120, 16), torch.nn.Flatten(), torch.nn.Linear(16 * 5, 120))
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
x, y = torch.randint(0, 120, (256, 5)), torch.randint(0, 120, (256,))
losses = []
for _ in range(20):
    optimizer.zero_grad()
    loss = torch.nn.functional.cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print(f"PyTorch OK | loss {losses[0]:.3f} → {losses[-1]:.3f} | CUDA: {torch.cuda.is_available()}")
assert losses[-1] < losses[0], "LỖI: loss không giảm"

In [ ]:
# Khối requirements.txt theo các gói đang cài. Bỏ đuôi +cpu/+cu… của torch vì PyPI không có bản đó.
from datetime import date

print(f"# Đã thử trên Kaggle {date.today():%d/%m/%Y}, Python {PYTHON} (notebooks/00_env_check.ipynb)")
for name, pin in pins.items():
    version = (installed(name) or "").split("+")[0]
    note = "" if version == pin else f"  # LỆCH: requirements.txt đang chốt {pin}"
    print(f"{name}=={version}{note}" if version else f"# LỖI: {name} không cài được")

## Kết quả (chạy trên Kaggle, CPU, 25/11/2026, bản notebook trước khi thêm assert và đọc requirements.txt)

- **Python** 3.13.18, Linux x86_64 (glibc 2.39).
- **Có sẵn:** numpy 2.1.3, pandas 2.3.3, pyarrow 23.0.1, scipy 1.16.3, gensim 4.4.0, torch 2.11.0+cpu, pytest 8.4.2.
- **Phải cài thêm:** implicit 0.7.3 (`pip install`, cần bật Internet); cài được ngay, không cần vá.
- **Bộ test của repo:** 194 passed in 4.38s trên `main`.
- **ALS, Word2Vec, một bước train PyTorch:** chạy hết, không lỗi (Run All dừng ở ô lỗi đầu tiên, và ô cuối đã chạy).
- `requirements.txt` ở gốc repo chép từ ô trên.